In [ ]:
%pip install qiskit==1.2.4
%pip install qiskit-aer==0.15.1
%pip install pylatexenc==2.10

from qiskit import QuantumCircuit
from qiskit.converters import circuit_to_gate
from qiskit.visualization import array_to_latex
from qiskit.quantum_info import Operator
from qiskit.quantum_info import Statevector
from qiskit import transpile
from qiskit.providers.basic_provider import BasicSimulator
from qiskit.visualization import plot_histogram
from qiskit.circuit import ControlledGate
import math

In [ ]:
# The aim of the assignment is to simulate the BB84 key distribution protocol.

# This notebook is for a simulation of the protocol without an attacker.



## Helper Function: Quantum Random Bit Generator

As specified, we will generate random bits by measuring a suitable quantum state, like $|+\rangle = \frac{1}{\sqrt{2}}(|0\rangle + |1\rangle)$. This ensures the randomness comes from quantum mechanics itself, rather than a pseudo-random number generator.

In [ ]:
def generate_quantum_random_bits(num_bits):
    """
    Generates a list of random bits (0 or 1) using quantum measurement.
    Each bit is generated by measuring a |+> state.
    """
    random_bits = []
    for _ in range(num_bits):
        qc = QuantumCircuit(1, 1) # 1 qubit, 1 classical bit
        qc.h(0) # Apply Hadamard to create |+> state
        qc.measure(0, 0) # Measure in the computational basis

        simulator = BasicSimulator()
        compiled_circuit = transpile(qc, simulator)
        job = simulator.run(compiled_circuit, shots=1)
        result = job.result()
        counts = result.get_counts(qc)

        # The result will be either {'0': 1} or {'1': 1}
        if '0' in counts: # If '0' was measured
            random_bits.append(0)
        else:
            random_bits.append(1)

    return random_bits

# Example usage (can be removed later if not needed for direct display)
# num_bits_for_example = 10
# random_q_bits = generate_quantum_random_bits(num_bits_for_example)
# print(f"Generated {num_bits_for_example} quantum random bits: {random_q_bits}")

## BB84 Protocol - No Attacker Scenario

This section simulates the BB84 protocol between Alice and Bob without any eavesdropper.

### Alice's Part: Prepare and Send Qubits

Alice first generates her secret key bits and chooses a random basis for each bit. She then encodes each bit into a qubit according to her chosen basis.

In [ ]:
# --- Parameters for the simulation ---
KEY_LENGTH = 100 # Number of bits in Alice's initial key

# Alice generates her secret key bits (0s and 1s)
alice_key_bits = generate_quantum_random_bits(KEY_LENGTH)
print(f"Alice's secret key bits (first 10): {alice_key_bits[:10]}...")

# Alice chooses a random basis for each bit (0 for Z basis, 1 for X basis)
alice_bases = generate_quantum_random_bits(KEY_LENGTH)
print(f"Alice's chosen bases (first 10): {alice_bases[:10]}... (0=Z, 1=X)")

# Alice encodes her bits into qubits
quantum_channel_qubits = [] # This will store the prepared qubits
for i in range(KEY_LENGTH):
    qc = QuantumCircuit(1, 1) # 1 qubit, 1 classical bit for potential measurement later (not here)
    bit = alice_key_bits[i]
    basis = alice_bases[i]

    if bit == 1: # If the bit is 1, apply X gate
        qc.x(0)

    if basis == 1: # If basis is X, apply Hadamard to rotate to X basis
        qc.h(0)

    quantum_channel_qubits.append(qc) # Add the prepared qubit circuit to the channel

print(f"\nAlice has prepared {len(quantum_channel_qubits)} qubits for sending.")
# Displaying the circuit for the first qubit as an example
# print("\nExample circuit for first qubit:")
# print(quantum_channel_qubits[0].draw(output='text'))

### Quantum Channel (Simulated)

In a real-world scenario, these qubits would be sent over an optical fiber. In our simulation, they are simply passed from Alice's preparation to Bob's measurement.

In [ ]:
# The 'quantum_channel_qubits' list implicitly represents the qubits traveling through the channel.

### Bob's Part: Receive and Measure Qubits

Bob receives the qubits and randomly chooses a basis for each measurement. He then measures each qubit and records the result.

In [ ]:
# Bob generates his random bases for measurement
bob_bases = generate_quantum_random_bits(KEY_LENGTH)
print(f"\nBob's chosen bases (first 10): {bob_bases[:10]}... (0=Z, 1=X)")

# Bob measures the received qubits
bob_measured_bits = []
simulator = BasicSimulator()

for i in range(KEY_LENGTH):
    qc_to_measure = quantum_channel_qubits[i] # Get Alice's prepared circuit
    bob_basis = bob_bases[i]

    # Create a new circuit for measurement, adding to Alice's original preparation
    # A new circuit is created to avoid modifying Alice's original preparation,
    # but we need to ensure the qubit state is carried over.
    # For simplicity in this simulation, we'll assume the state is 'passed' and Bob starts a fresh measurement circuit.

    # To properly simulate, we need to apply Bob's basis transformation to Alice's prepared state.
    # We extract the state from Alice's prepared circuit and then apply Bob's operations.

    # Get the state prepared by Alice
    # This part requires running Alice's circuit to get the final state.
    # In a practical simulation, we would create a new circuit and append gates to the *same* qubit.
    # For this simplified sequential program, we'll modify a copy of Alice's circuit for Bob's measurement.

    # Copy Alice's circuit for Bob's operations
    bob_qc = qc_to_measure.copy_empty_like()
    for instruction in qc_to_measure.data: # Reconstruct Alice's operations
        bob_qc.append(instruction)

    if bob_basis == 1: # If Bob chooses X basis, apply Hadamard
        bob_qc.h(0)

    bob_qc.measure(0, 0) # Measure in the computational basis

    # Simulate the measurement
    compiled_circuit = transpile(bob_qc, simulator)
    job = simulator.run(compiled_circuit, shots=1)
    result = job.result()
    counts = result.get_counts(bob_qc)

    # Extract the measured bit
    if '0' in counts:
        bob_measured_bits.append(0)
    else:
        bob_measured_bits.append(1)

print(f"Bob's measured bits (first 10): {bob_measured_bits[:10]}...")

### Classical Communication: Basis Sifting

Alice and Bob publicly announce their chosen bases. They keep only the bits where their bases matched. These form the raw key.

In [ ]:
alice_sifted_key = []
bob_sifted_key = []

for i in range(KEY_LENGTH):
    if alice_bases[i] == bob_bases[i]:
        alice_sifted_key.append(alice_key_bits[i]) # Alice keeps her original bit
        bob_sifted_key.append(bob_measured_bits[i]) # Bob keeps his measured bit

print(f"\nLength of raw key after sifting: {len(alice_sifted_key)} bits")
print(f"Alice's sifted key (first 10): {alice_sifted_key[:10]}...")
print(f"Bob's sifted key (first 10): {bob_sifted_key[:10]}...")

### Classical Communication: Error Checking

Alice and Bob compare a subset of their raw key bits to check for errors (or eavesdropping in a more complex scenario). Since there's no attacker, we expect a perfect match.

In [ ]:
# Compare the sifted keys
error_count = 0
for i in range(len(alice_sifted_key)):
    if alice_sifted_key[i] != bob_sifted_key[i]:
        error_count += 1

print(f"\nNumber of mismatches between Alice's and Bob's sifted keys: {error_count}")

if error_count == 0:
    print("Alice and Bob successfully established a shared secret key with no errors.")
    print("Shared secret key (first 10 bits):", alice_sifted_key[:10])
else:
    print(f"Errors detected. The error rate is {error_count / len(alice_sifted_key):.2%}.")
    print("This indicates a potential issue (e.g., noise or an attacker).")
